# Zero-shot vs few-shot

**Session 2 · small model (`llama3.2:3b`) vs big model (`gpt-oss:120b-cloud`)**

The folklore says few-shot always wins. Measure it. On a modern small model doing a familiar
task (support-ticket routing), adding examples often does **not** clear the noise — `compare()`
says INCONCLUSIVE. So: start zero-shot, and add examples only when the number tells you to.
Few-shot earns its keep on unfamiliar label schemes, strict output formats, and specific edge
cases — the "your turn" section builds one.

In [ ]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
from utils import ask, SMALL_MODEL, BIG_MODEL
from eval import load_cases, compare, sweep_models, exact


### Worked example

Route support tickets to `billing` / `bug` / `other` over the 40-case set, with a tolerant
scorer (the label appears anywhere in the reply). Zero-shot vs a 5-example few-shot prompt,
`repeats=3`.

In [ ]:
cases = load_cases("../eval/datasets/support_tickets.jsonl")
LABELS = ["billing", "bug", "other"]

def clean(out):
    out = out.strip().lower()
    return next((lab for lab in LABELS if lab in out), out)

ZERO = ('Classify the support ticket as billing, bug, or other.\n'
        'Reply with ONE lowercase word.\n\nTicket: "{t}" ->')

FEW = ('Classify the support ticket as billing, bug, or other.\n'
       'Reply with ONE lowercase word.\n\n'
       'Ticket: "I cannot log in since the update" -> bug\n'
       'Ticket: "Refund me for the double charge" -> billing\n'
       'Ticket: "What are your office hours?" -> other\n'
       'Ticket: "How do I update my payment card?" -> other\n'
       'Ticket: "The totals on the dashboard are wrong" -> bug\n\n'
       'Ticket: "{t}" ->')

def make(template, model):
    return lambda t: clean(ask(template.format(t=t), model=model))

print("small model:")
compare(cases, make(ZERO, SMALL_MODEL), make(FEW, SMALL_MODEL),
        labels=("zero-shot", "few-shot"), scorer=exact, repeats=3)


### Same examples, bigger model

`sweep_models` runs each prompt on both models. The big model's zero-shot already scores near
its few-shot number, so the examples buy little there — spend that prompt budget elsewhere.

In [ ]:
for shot, tmpl in [("zero-shot", ZERO), ("few-shot", FEW)]:
    print(f"\n{shot}:")
    sweep_models(cases, lambda m, t=tmpl: make(t, m), [SMALL_MODEL, BIG_MODEL],
                 scorer=exact, repeats=2)


## Your turn - vary the example

1. **Make few-shot win.** Invent a label scheme the model can't guess: priority `P1`/`P2`/`P3`
   with a specific rule ("P1 = paying customer blocked"). Zero-shot will flail; a few examples
   should pin the rule. Re-run `compare()` — now is it REAL?
2. Score on the **raw** output with `scorer=exact` and no `clean()`. Does few-shot help or hurt
   parseability here? (It can go either way with a `->` continuation prompt — measure.)
3. Record the small-model and big-model zero-vs-few gaps, with verdicts, in your commit message.